Objetivo del proyecto

El objetivo de este proyecto es analizar el comportamiento de los precios de alojamientos en Airbnb y su relación con diferentes variables como:
- tipo de alojamiento
- barrio
- capacidad del alojamiento
- número de reseñas
- disponibilidad

A través de este análisis exploratorio de datos (EDA) se busca identificar patrones que permitan comprender qué factores influyen en el precio de las propiedades.

In [51]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

In [52]:
# Carga del dataset

data_calendar = pd.read_csv('../data/calendar.csv')
data_listings = pd.read_csv('../data/listings.csv', low_memory=False)
data_reviews = pd.read_csv('../data/reviews.csv')


1. Se comenzará analizando el df principal

In [53]:
# Información general del dataset donde estan las propiedades de los alojamientos
data_listings.info()


<class 'pandas.DataFrame'>
RangeIndex: 23729 entries, 0 to 23728
Columns: 106 entries, id to reviews_per_month
dtypes: float64(23), int64(21), str(62)
memory usage: 19.2 MB


In [54]:
# Tamaño del dataset
data_listings.shape


(23729, 106)

In [55]:
# nombres de las columnas del dataset
sorted(data_listings.columns)

['access',
 'accommodates',
 'amenities',
 'availability_30',
 'availability_365',
 'availability_60',
 'availability_90',
 'bathrooms',
 'bed_type',
 'bedrooms',
 'beds',
 'calculated_host_listings_count',
 'calculated_host_listings_count_entire_homes',
 'calculated_host_listings_count_private_rooms',
 'calculated_host_listings_count_shared_rooms',
 'calendar_last_scraped',
 'calendar_updated',
 'cancellation_policy',
 'city',
 'cleaning_fee',
 'country',
 'country_code',
 'description',
 'experiences_offered',
 'extra_people',
 'first_review',
 'guests_included',
 'has_availability',
 'host_about',
 'host_acceptance_rate',
 'host_has_profile_pic',
 'host_id',
 'host_identity_verified',
 'host_is_superhost',
 'host_listings_count',
 'host_location',
 'host_name',
 'host_neighbourhood',
 'host_picture_url',
 'host_response_rate',
 'host_response_time',
 'host_since',
 'host_thumbnail_url',
 'host_total_listings_count',
 'host_url',
 'host_verifications',
 'house_rules',
 'id',
 'instan

In [56]:
# Visualización de las primeras filas del dataset
data_listings.head()

,id,listing_url,scrape_id,last_scraped,name,summary,space,description,experiences_offered,neighborhood_overview,...,instant_bookable,is_business_travel_ready,cancellation_policy,require_guest_profile_picture,require_guest_phone_verification,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,11508,https://www.airbnb.com/rooms/11508,20200426042522,2020-04-26,Amazing Luxurious Apt-Palermo Soho,NaN,LUXURIOUS NEW APT: 1 BDRM- POOL/ GYM/ SPA/ 24-...,LUXURIOUS NEW APT: 1 BDRM- POOL/ GYM/ SPA/ 24-...,none,AREA: PALERMO SOHO Minutes walking distance fr...,...,f,f,strict_14_with_grace_period,f,f,1,1,0,0,0.27
1,12463,https://www.airbnb.com/rooms/12463,20200426042522,2020-04-26,Room in Recoleta - awesome location,My apartment is centrally located in Recoleta ...,This is a very comfortable pull-out sofa in th...,My apartment is centrally located in Recoleta ...,none,It's near the school of medicine so the street...,...,f,f,moderate,f,f,1,0,1,0,0.16
2,13095,https://www.airbnb.com/rooms/13095,20200426042522,2020-04-26,Standard Room at Palermo Viejo B&B,Palermo Viejo B&B is a typical home in one of ...,Standard room : Palermo Viejo Bed & Breakfast ...,Palermo Viejo B&B is a typical home in one of ...,none,NaN,...,f,f,strict_14_with_grace_period,f,f,7,0,7,0,0.06
3,13096,https://www.airbnb.com/rooms/13096,20200426042522,2020-04-26,Standard Room in Palermo Viejo B&B,Palermo Viejo B&B is a typical home in one of ...,Palermo Viejo Bed & Breakfast is located in a ...,Palermo Viejo B&B is a typical home in one of ...,none,NaN,...,f,f,strict_14_with_grace_period,f,f,7,0,7,0,NaN
4,13097,https://www.airbnb.com/rooms/13097,20200426042522,2020-04-26,Standard Room at Palermo Viejo B&B,Palermo Viejo B&B is a typical home in one of ...,Palermo Viejo Bed & Breakfast is located in a ...,Palermo Viejo B&B is a typical home in one of ...,none,NaN,...,f,f,strict_14_with_grace_period,f,f,7,0,7,0,1.89


In [57]:
#eliminar duplicados
data_listings = data_listings.drop_duplicates()

In [58]:
# Análisis de valores nulos y ordenamiento de los mismos en orden descendente

nulos = data_listings.isnull().sum().sort_values(ascending=False)
nulos.head(30)

thumbnail_url                   23729
medium_url                      23729
neighbourhood_group_cleansed    23729
xl_picture_url                  23729
license                         23729
jurisdiction_names              23722
square_feet                     23346
weekly_price                    21188
monthly_price                   21151
notes                           16007
access                          13525
house_rules                     13078
host_about                      10000
interaction                      9824
transit                          9221
security_deposit                 8195
neighborhood_overview            7665
space                            7194
review_scores_value              6909
review_scores_communication      6908
review_scores_cleanliness        6907
review_scores_location           6907
review_scores_checkin            6907
review_scores_accuracy           6907
review_scores_rating             6890
host_response_rate               6886
host_respons

In [59]:
# columnas a eliminar

cols_drop = [
    "weekly_price",
    "monthly_price",
    'neighbourhood_group_cleansed',
    'license',
    'jurisdiction_names',
    'notes',
    'access',
    'interaction',
    'house_rules',
    'host_about',
    'transit',
    'space',
    'square_feet',
    'neighborhood_overview',
    'host_name',
    'host_location',
    'host_neighbourhood',
    'host_verifications',
    'host_has_profile_pic',
    "scrape_id",
    "last_scraped",
    "name",
    "description",
    "summary",
    "first_review",
    "last_review",
    "zipcode",
    "experiences_offered",
    "street",
    "smart_location",
    "market",
    "country_code",
    "country",
    "calendar_updated",
    "calendar_last_scraped",
    "has_availability",
    "is_location_exact",
    "requires_license",
    "require_guest_profile_picture",
    "require_guest_phone_verification",
    "is_business_travel_ready",
    "amenities"
]

data_listings = data_listings.drop(columns=[c for c in cols_drop if c in data_listings.columns])


- Se eliminaron columnas con alto porcentaje de valores faltantes (>60%)
- Se eliminaron campos descriptivos de texto libre que no aportan valor al análisis estructurado

In [60]:
# Eliminacion de columnas que contienen la palabra "url"
columnas_url = [col for col in data_listings.columns if "url" in col]
data_listings = data_listings.drop(columns=columnas_url) 


In [61]:
# nombres de columnas que incluyen la palabra "host"
columnas_host = [col for col in data_listings.columns if "host" in col]
print(columnas_host)

['host_id', 'host_since', 'host_response_time', 'host_response_rate', 'host_acceptance_rate', 'host_is_superhost', 'host_listings_count', 'host_total_listings_count', 'host_identity_verified', 'calculated_host_listings_count', 'calculated_host_listings_count_entire_homes', 'calculated_host_listings_count_private_rooms', 'calculated_host_listings_count_shared_rooms']


In [62]:
(data_listings['host_listings_count'] == data_listings['host_total_listings_count']).sum()

np.int64(23726)

In [63]:
# eliminacion de columnas repetidas de "host"

columnas_calculated = [col for col in data_listings.columns if "calculated_host" in col]
data_listings = data_listings.drop(columns=columnas_calculated)

data_listings = data_listings.drop('host_listings_count', axis=1)

- Se eliminaron columnas relacionadas a URLs e imágenes sin relevancia analítica
- Se eliminaron columnas calculadas en base a otras columnas, se repite la información

In [64]:
# Análisis de valores nulos en orden descendente después de eliminar las columnas
data_listings.isnull().sum().sort_values(ascending=False)

security_deposit               8195
review_scores_value            6909
review_scores_communication    6908
review_scores_accuracy         6907
review_scores_location         6907
review_scores_cleanliness      6907
review_scores_checkin          6907
review_scores_rating           6890
host_response_time             6886
host_response_rate             6886
cleaning_fee                   6785
reviews_per_month              6507
host_acceptance_rate           5199
city                            587
state                           183
beds                            167
bathrooms                        62
bedrooms                         43
host_since                        3
host_is_superhost                 3
host_total_listings_count         3
host_identity_verified            3
accommodates                      0
latitude                          0
neighbourhood_cleansed            0
neighbourhood                     0
host_id                           0
id                          

In [65]:
# Análisis de porcentaje de valores nulos en orden descendente

(data_listings.isna().mean() * 100).sort_values(ascending=False)

security_deposit               34.535800
review_scores_value            29.116271
review_scores_communication    29.112057
review_scores_accuracy         29.107843
review_scores_location         29.107843
review_scores_cleanliness      29.107843
review_scores_checkin          29.107843
review_scores_rating           29.036200
host_response_time             29.019343
host_response_rate             29.019343
cleaning_fee                   28.593704
reviews_per_month              27.422142
host_acceptance_rate           21.909899
city                            2.473766
state                           0.771208
beds                            0.703780
bathrooms                       0.261284
bedrooms                        0.181213
host_since                      0.012643
host_is_superhost               0.012643
host_total_listings_count       0.012643
host_identity_verified          0.012643
accommodates                    0.000000
latitude                        0.000000
neighbourhood_cl

In [66]:
# Imputación de valores nulos en variables economomicas con el valor 0, suponiendo que no se cobra por ese concepto

cols_economicas = ["cleaning_fee", "security_deposit", "extra_people"]
data_listings[cols_economicas] = data_listings[cols_economicas].fillna(0)


In [67]:
# reemplazo de valores nulos en la columnas de caracteristicas de la propiedad con la mediana de las mismas
data_listings["bathrooms"] = data_listings["bathrooms"].fillna(data_listings["bathrooms"].median())
data_listings["bedrooms"] = data_listings["bedrooms"].fillna(data_listings["bedrooms"].median())
data_listings["beds"] = data_listings["beds"].fillna(data_listings["beds"].median())

In [68]:
# cambio de formato de las columnas de dinero a formato numérico
columnas_dinero = ["price", "cleaning_fee", "security_deposit", "extra_people"]
for col in columnas_dinero:
    data_listings[col] = data_listings[col].replace(r'[\$,]', '', regex=True).astype(float)

In [69]:
# cambio de formato de las columnas de porcentaje a formato numérico

cols_percent = ['host_response_rate', 'host_acceptance_rate']

for col in cols_percent:
    if col in data_listings.columns:
        data_listings[col] = (
            data_listings[col]
            .str.replace('%','')
            .astype(float)
        )


In [70]:
# cambio de formato de las columnas de fecha a formato datetime
cols_fecha = ['host_since', 'first_review', 'last_review'] 
for col in cols_fecha:
    if col in data_listings.columns:
        data_listings[col] = pd.to_datetime(data_listings[col])


In [71]:
# cambio de formato de las columnas booleanas a formato numérico

cols_bool = [
    'host_is_superhost',
    'instant_bookable'
]
for col in cols_bool:
    if col in data_listings.columns:
        data_listings[col] = data_listings[col].map({'t':1,'f':0})


In [72]:
# Imputación de valores nulos en la columna "host_is_superhost" con el valor 0, suponiendo que no es superhost

data_listings["host_is_superhost"] = data_listings["host_is_superhost"].fillna(0)

In [73]:
# Imputación de valores nulos en la columna "host_response_rate" con el valor 0, suponiendo que no se ha respondido a ninguna consulta

data_listings["host_response_rate"] = data_listings["host_response_rate"].fillna(0)


In [74]:
# Imputación de valores nulos en la columna "host_response_time" con el valor "unknown", suponiendo que no se ha registrado el tiempo de respuesta

data_listings["host_response_time"] = data_listings["host_response_time"].fillna("unknown")

In [75]:
# Imputación de valores nulos en las columnas "reviews_per_month" y "host_acceptance_rate" con el valor 0, suponiendo que no se han recibido reseñas ni se ha aceptado ninguna reserva
data_listings[['reviews_per_month', "host_acceptance_rate"]] = data_listings[['reviews_per_month', "host_acceptance_rate"]].fillna(0)

In [76]:
# Imputacion de valores nulos en review_scores con el valor 0, suponiendo que no se han recibido reseñas o no se han calificado
cols_review_scores = [col for col in data_listings.columns if col.startswith("review_scores")]
data_listings[cols_review_scores] = data_listings[cols_review_scores].fillna(0)

In [77]:
# Revisión final de valores nulos 

data_listings.isnull().sum().sort_values(ascending=False)


city                           587
state                          183
host_since                       3
host_identity_verified           3
host_total_listings_count        3
host_id                          0
id                               0
host_is_superhost                0
host_acceptance_rate             0
host_response_time               0
host_response_rate               0
neighbourhood_cleansed           0
neighbourhood                    0
latitude                         0
longitude                        0
property_type                    0
room_type                        0
accommodates                     0
bathrooms                        0
bedrooms                         0
beds                             0
bed_type                         0
price                            0
security_deposit                 0
cleaning_fee                     0
guests_included                  0
extra_people                     0
minimum_nights                   0
maximum_nights      

Se realizó una limpieza preliminar seleccionando variables relevantes, convirtiendo tipos de datos, imputando valores faltantes en variables no críticas y eliminando registros incompletos únicamente en campos esenciales.

2. Limpieza de datos de calendar

In [78]:
data_calendar.info()

<class 'pandas.DataFrame'>
RangeIndex: 8661286 entries, 0 to 8661285
Data columns (total 7 columns):
 #   Column          Dtype  
---  ------          -----  
 0   listing_id      int64  
 1   date            str    
 2   available       str    
 3   price           str    
 4   adjusted_price  str    
 5   minimum_nights  float64
 6   maximum_nights  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 462.6 MB


In [79]:
data_calendar.head()

,listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,50778,2020-04-26,f,"$2,655.00","$2,655.00",5.0,1125.0
1,133654,2020-04-27,t,"$1,150.00","$1,150.00",4.0,1125.0
2,133654,2020-04-28,t,"$1,150.00","$1,150.00",4.0,1125.0
3,133654,2020-04-29,t,"$1,150.00","$1,150.00",4.0,1125.0
4,133654,2020-04-30,t,"$1,150.00","$1,150.00",4.0,1125.0


In [80]:
# Limpieza de la columna "price" en el dataset calendar para convertirla a formato numérico

data_calendar["price"] = (data_calendar["price"].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float))

In [81]:
# Limpieza de la columna "date" en el dataset calendar para convertirla a formato datetime

data_calendar["date"] = pd.to_datetime(data_calendar["date"])

In [82]:
# Limpieza de la columna "available" en el dataset calendar para convertirla a formato numérico binario (1 para "t" y 0 para "f")

data_calendar["available"] = data_calendar["available"].map({"t": 1, "f": 0})


In [83]:
# Verificación de valores nulos en el dataset calendar después de la limpieza

data_calendar.isnull().sum()


listing_id          0
date                0
available           0
price               0
adjusted_price      0
minimum_nights    105
maximum_nights    105
dtype: int64

In [84]:
# Cálculo de la tasa de ocupación para cada propiedad utilizando la columna "available" del dataset calendar

occupancy_rate = (data_calendar.groupby("listing_id")["available"].apply(lambda x: 1 - x.mean()))


In [85]:
# Cálculo del precio promedio por noche para cada propiedad utilizando la columna "price" del dataset calendar

avg_price = data_calendar.groupby("listing_id")["price"].mean()

In [86]:
#Creacion de un nuevo dataframe con el resumen de las variables calculadas para cada propiedad

calendar_summary = data_calendar.groupby("listing_id").agg(
    occupancy_rate=("available", lambda x: 1 - x.mean()),
    avg_price=("price","mean"),
    availability_rate=("available","mean"),
    min_nights=("minimum_nights","mean")
).reset_index()

calendar_summary["occupancy_rate"] = 1 - calendar_summary["availability_rate"]

In [87]:
# Unión del dataframe de propiedades con el resumen del dataset

df = data_listings.merge(
    calendar_summary,
    left_on="id",
    right_on="listing_id",
    how="left"
)

Debido al gran volumen del dataset calendar se realizó un join por listing_id para obtener variables de precio promedio y tasa de ocupación con el fin de reducir el tamaño del dataset antes de integrarlo con la información de propiedades

3. Análisis de Dataset reviews

In [88]:
data_reviews.info()

<class 'pandas.DataFrame'>
RangeIndex: 387099 entries, 0 to 387098
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   listing_id     387099 non-null  int64
 1   id             387099 non-null  int64
 2   date           387099 non-null  str  
 3   reviewer_id    387099 non-null  int64
 4   reviewer_name  387099 non-null  str  
 5   comments       386923 non-null  str  
dtypes: int64(3), str(3)
memory usage: 17.7 MB


In [89]:
data_reviews.head()

,listing_id,id,date,reviewer_id,reviewer_name,comments
0,11508,1615861,2012-07-02,877808,Charlie,Amazing place!\r\n\r\nLocation: short walk to ...
1,11508,3157005,2012-12-26,656077,Shaily,Really enjoyed Candela's recommendations and q...
2,11508,3281011,2013-01-05,2835998,Michiel,Candela and her colleague were very attentive ...
3,11508,6050019,2013-07-28,4600436,Tara,"The apartment was in a beautiful, modern build..."
4,11508,9328455,2013-12-22,3130017,Simon,My stay at Candela's apartment was very enjoya...


In [90]:
# Agregación de información relevante del dataset reviews

reviews_summary = data_reviews.groupby("listing_id").agg(
    reviews_count=("id", "count"),
    first_review=("date", "min"),
    last_review=("date", "max")
).reset_index()

In [91]:
# Cálculo de los años activos de cada propiedad en base a la fecha del primer y último review
reviews_summary["years_active"] = (
    pd.to_datetime(reviews_summary["last_review"]) -
    pd.to_datetime(reviews_summary["first_review"])
).dt.days / 365

In [92]:
# Unión del dataframe de propiedades con el resumen del dataset reviews
df = data_listings.merge(
    reviews_summary,
    left_on="id",
    right_on="listing_id",
    how="left"
)

El dataset de reviews contiene muchísimos comentarios históricos, para optimizar el análisis se realizó un join por listing_id generando variables como cantidad de reviews y periodo de actividad de cada propiedad

In [93]:
df.columns

Index(['id', 'host_id', 'host_since', 'host_response_time',
       'host_response_rate', 'host_acceptance_rate', 'host_is_superhost',
       'host_total_listings_count', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'city', 'state', 'latitude', 'longitude',
       'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms',
       'beds', 'bed_type', 'price', 'security_deposit', 'cleaning_fee',
       'guests_included', 'extra_people', 'minimum_nights', 'maximum_nights',
       'minimum_minimum_nights', 'maximum_minimum_nights',
       'minimum_maximum_nights', 'maximum_maximum_nights',
       'minimum_nights_avg_ntm', 'maximum_nights_avg_ntm', 'availability_30',
       'availability_60', 'availability_90', 'availability_365',
       'number_of_reviews', 'number_of_reviews_ltm', 'review_scores_rating',
       'review_scores_accuracy', 'review_scores_cleanliness',
       'review_scores_checkin', 'review_scores_communication',
       'review_scores

In [94]:
# Eliminacion de columnas con "minimum" y "maximum" que no aportan valor al análisis y pueden generar ruido en el modelo

col_min_max={"minimum_minimum_nights",
    "maximum_minimum_nights",
    "minimum_maximum_nights",
    "maximum_maximum_nights",
    "minimum_nights_avg_ntm",
    "maximum_nights_avg_ntm"
    }

df = df.drop(columns=col_min_max)

In [95]:
# Eliminacion de columnas con "reviews" que aportan a "reviews rating" y pueden generar ruido en el modelo

col_reviews = {
    "review_scores_accuracy",
    "review_scores_cleanliness",
    "review_scores_checkin",
    "review_scores_communication",
    "review_scores_location",
    "review_scores_value"
}

df = df.drop(columns=col_reviews)

In [96]:
# eliminacion de columnas con informacion repetida

col_repetidas = {"neighbourhood", "state", "availability_30", "availability_60", "availability_90"}
df = df.drop(columns=col_repetidas)

In [97]:
# Creación de la columna "host_years" calculando la antigüedad del anfitrión en años a partir de la columna "host_since"
df["host_years"] = 2026 - df["host_since"].dt.year

In [98]:
# Creación de la columna "occupancy_rate" calculando la tasa de ocupación a partir de la columna "availability_365"
df['occupancy_rate'] = (365 - df['availability_365']) / 365

In [99]:
# Creación de la columna "estimated_revenue" calculando el ingreso estimado por año a partir del precio promedio por noche y la tasa de ocupación
df["estimated_revenue"] = df["price"] * df["occupancy_rate"] * 365

In [100]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 23729 entries, 0 to 23728
Data columns (total 42 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   id                         23729 non-null  int64         
 1   host_id                    23729 non-null  int64         
 2   host_since                 23726 non-null  datetime64[us]
 3   host_response_time         23729 non-null  str           
 4   host_response_rate         23729 non-null  float64       
 5   host_acceptance_rate       23729 non-null  float64       
 6   host_is_superhost          23729 non-null  float64       
 7   host_total_listings_count  23726 non-null  float64       
 8   host_identity_verified     23726 non-null  str           
 9   neighbourhood_cleansed     23729 non-null  str           
 10  city                       23142 non-null  str           
 11  latitude                   23729 non-null  float64       
 12  longitude      

In [101]:
# Guardado del DataFrame limpio en archivo CSV

df.to_csv("../data/listings_clean.csv", index=False)